In [1]:
import pandas as pd

In [2]:
import os
from os.path import join
from tqdm import tqdm

In [3]:
base_path = r"C:\Users\tre_i\Downloads\dataset"

In [26]:
import pickle


with open("files/FIW_buffalo_sc_faces_info.pkl", "rb") as f:
    FIW_buffalo_sc_faces_info = pickle.load(f)
photo_files_paths = list(FIW_buffalo_sc_faces_info.keys())

In [34]:
import os


photo_files_dict = {}
for el in photo_files_paths:
    family, person = el.split(os.sep)[-3:-1]
    photo_files_dict[(family, person)] = photo_files_dict.get((family, person), []) + [el]

In [44]:
family_dfs = {}

root = join(base_path, 'Train/train-faces')
for family in tqdm(os.listdir(root)):
    family_df_path = join(root, family, "mid.csv")
    family_df = pd.read_csv(family_df_path, index_col=0)
    family_dfs[family] = family_df

100%|██████████| 571/571 [00:00<00:00, 685.69it/s]


In [45]:
family

'F1018'

In [46]:
family_df

,1,2,3,4,5,6,7,8,Name,Gender
MID,,,,,,,,,,
1,0,5,4,4,4,1,4,4,holly,f
2,5,0,4,4,0,0,0,0,rodney,m
3,1,1,0,2,0,3,2,2,ryan,f
4,1,1,2,0,4,3,2,2,robinson,m
5,1,0,0,1,0,3,2,2,rodney,m
6,4,0,6,6,6,0,6,6,dolores robinson,f
7,1,0,2,2,2,3,0,2,roman,m
8,1,0,2,2,2,3,2,0,rodney jr,m


In [48]:
with open('files/FIW_family_dfs.pkl', 'wb') as f:
    pickle.dump(family_dfs, f)

In [39]:
families_list_json = []
gender_replace = {"f": "Female", "m": "Male"}
relation_dict = {1: "parent", 4: "child",  5: "spouse"}

for family, family_df in family_dfs.items():
    family_members = {}
    indexes = family_df.index
    cols = family_df.columns
    for col in cols:
        try:
            if int(col) not in set(indexes):
                family_df = family_df.drop(columns=[col]).copy()
        except:
            continue
    for index in family_df.index:
        person_id = f"{family}_{index}"
        row = family_df.loc[index]
        name = row['Name']
        gender = row['Gender']
        gender = gender_replace[gender]
        relatives = []
        for col in row.keys():
            try:
                other_person_index = int(col)
            except:
                continue
            if other_person_index != index:
                other_person_id = f"{family}_{other_person_index}"
                relation_type = relation_dict.get(row[col], None)
                if relation_type is not None:
                    relatives.append({
                        "person_id": other_person_id,
                        "relationType": relation_type,
                    })

        photo_paths = photo_files_dict.get((family, f"MID{index}"), [])

        family_members[person_id] = {
            "person_id": person_id,
            "family_id": family,
            "gender": gender,
            "name": name,
            "surname": None,
            "birthdate": {
                "day": None,
                "month": None,
                "year": None
            },
            "middleName": None,
            "birthplace": None,
            "relatives": relatives,
            "photo_paths": photo_paths
        }

    #need to add parents for siblings who don't have parents in the dataset, not ideal way in case of half siblings
    for index in family_df.index:
        person_id = f"{family}_{index}"
        row = family_df.loc[index]
        siblings = [person_id]
        for col in row.keys():
            try:
                other_person_index = int(col)
            except:
                continue

            if other_person_index != index and row[col] == 2:
                other_person_id = f"{family}_{other_person_index}"
                siblings.append(other_person_id)

        if len(siblings) > 1:
            siblings_parents = []
            for sibling in siblings:
                sibling_dict = family_members[sibling]
                sibling_relatives = sibling_dict["relatives"]
                sibling_parents = {el["person_id"] for el in sibling_relatives if el["relationType"] == "parent"}
                siblings_parents.append(sibling_parents)
            common_parents = set.intersection(*siblings_parents)
            if len(common_parents) == 0:
                additional_parents = [el for el in family_members.keys() if "additional_parent" in el]
                if len(additional_parents) > 0:
                    max_additional_parent = max([int(el[el.rfind("_") + 1:]) for el in additional_parents])
                else:
                    max_additional_parent = 0
                parent_id = f"{family}_additional_parent_number_{max_additional_parent + 1}"
                relatives = [{
                        "person_id": el,
                        "relationType": "child",
                    } for el in siblings]
                parent = {
                    "person_id": parent_id,
                    "family_id": family,
                    "gender": None,
                    "name": None,
                    "surname": None,
                    "birthdate": {
                        "day": None,
                        "month": None,
                        "year": None
                    },
                    "middleName": None,
                    "birthplace": None,
                    "relatives": relatives,
                    "photo_paths": []
                }
                family_members[parent_id] = parent

                for sibling in siblings:
                    family_members[sibling]["relatives"].append({
                        "person_id": parent_id,
                        "relationType": "parent",
                    })

    families_list_json += list(family_members.values())

In [42]:
import json


with open("files/FIW_family_members.json", "w", encoding="utf-8") as f:
    json.dump(families_list_json, f, ensure_ascii=False, indent=4)